# 08 随机森林 Random Forest

依赖安装说明：`pip install numpy matplotlib scikit-learn`

随机森林是很多棵决策树的集成。每棵树只看一部分样本和一部分特征，最后投票或平均，通常比单棵树更稳定。


## 1. 数学逻辑

随机森林使用 bagging：

1. 从训练集有放回抽样，得到很多 bootstrap 数据集。
2. 每个数据集训练一棵树。
3. 分类时投票，回归时平均。

分类预测可以写成：

$$\hat y = \text{mode}\{T_1(x), T_2(x), \cdots, T_B(x)\}$$

随机性降低了树之间的相关性，平均后方差下降。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(42)
X, y = make_moons(n_samples=300, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


In [ ]:
# 从零感受 bagging：训练多棵浅树，然后投票
rng = np.random.default_rng(42)
trees = []
for b in range(25):
    ids = rng.integers(0, len(X_train), size=len(X_train))
    tree = DecisionTreeClassifier(max_depth=4, max_features=1, random_state=b)
    tree.fit(X_train[ids], y_train[ids])
    trees.append(tree)

all_preds = np.array([tree.predict(X_test) for tree in trees])
bagging_pred = []
for col in all_preds.T:
    bagging_pred.append(Counter(col).most_common(1)[0][0])
bagging_pred = np.array(bagging_pred)

print('单棵树 accuracy:', round(accuracy_score(y_test, trees[0].predict(X_test)), 3))
print('bagging accuracy:', round(accuracy_score(y_test, bagging_pred), 3))


In [ ]:
model = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('RandomForest accuracy:', round(accuracy_score(y_test, pred), 3))
print('feature importance:', np.round(model.feature_importances_, 3))

xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 180),
                     np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 180))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = model.predict(grid).reshape(xx.shape)
plt.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
plt.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap='coolwarm', edgecolor='k', s=24)
plt.title('随机森林决策边界')
plt.show()


## 2. 常见误区

- 随机森林通常强于单棵树，但可解释性会下降。
- 树很多不一定过拟合更严重，但训练和预测会更慢。
- 特征重要性会偏向取值多、切分机会多的特征。

## 3. 小实验

- 改 `n_estimators`，观察稳定性。
- 改 `max_depth`，观察过拟合。
- 对比单棵树和随机森林的决策边界。
